In [9]:
import sys
import numpy as np
import pyqtgraph as pg
from PyQt6 import QtWidgets, QtCore

# The Magic: NVIDIA GPU Drop-in Replacements
import cupy as cp
import cupyx.scipy.fft as cufft

class GPE_GPU_MVP:
    def __init__(self):
        # --- Physics Parameters ---
        self.N = 256          # We can handle 256x256 easily now!
        self.L = 30.0
        self.dt = 0.001
        self.steps_per_frame = 50  # Let the GPU do 50 math steps per 1 screen draw
        
        self.setup_physics()
        self.setup_gui()
        
        # Start the high-speed rendering loop
        self.timer = QtCore.QTimer()
        self.timer.timeout.connect(self.update_loop)
        self.timer.start(16) # Roughly 60 FPS target (16 ms)

    def setup_physics(self):
        # 1. Initialize Grids directly on the GPU VRAM
        x = cp.linspace(-self.L/2, self.L/2, self.N, endpoint=False, dtype=cp.float32)
        self.X, self.Y = cp.meshgrid(x, x)
        self.dx = float(x[1] - x[0])
        self.dx2 = self.dx**2

        kx = cufft.fftfreq(self.N, d=self.dx).astype(cp.float32) * 2 * cp.pi
        Kx, Ky = cp.meshgrid(kx, kx)
        self.K_sq = Kx**2 + Ky**2

        # Precompute kinetic operator on GPU
        self.exp_K = cp.exp(-1j * 0.5 * self.dt * self.K_sq).astype(cp.complex64)

        # 2. Create Two Colliding Wavepackets on GPU
        # Left packet moving right
        r1_sq = (self.X - -6)**2 + (self.Y - 1.5)**2
        psi1 = cp.exp(-0.5 * r1_sq / 2.5**2) * cp.exp(1j * (15.0 * self.X))
        
        # Right packet moving left
        r2_sq = (self.X - 6)**2 + (self.Y - -1.5)**2
        psi2 = cp.exp(-0.5 * r2_sq / 2.5**2) * cp.exp(1j * (-15.0 * self.X))

        self.psi = (psi1 + psi2).astype(cp.complex64)
        
        # Normalize
        norm = cp.sqrt(cp.sum(self.psi.real**2 + self.psi.imag**2) * self.dx2)
        self.psi /= norm
        
        # Cache max density to lock the color scaling
        self.max_dens = float(cp.max(self.psi.real**2 + self.psi.imag**2)) * 1.5

    def setup_gui(self):
        self.app = pg.mkQApp("2D GPE GPU Simulator")
        self.win = QtWidgets.QMainWindow()
        self.win.setWindowTitle("GPU Accelerated GPE (PyQtGraph + CuPy)")
        self.win.resize(1200, 600)
        
        # Main Layout
        central_widget = QtWidgets.QWidget()
        layout = QtWidgets.QHBoxLayout()
        central_widget.setLayout(layout)
        self.win.setCentralWidget(central_widget)

        # --- Left Panel: Minimal Controls ---
        control_panel = QtWidgets.QVBoxLayout()
        control_panel.setAlignment(QtCore.Qt.AlignmentFlag.AlignTop)
        
        lbl = QtWidgets.QLabel("GPU Controls")
        lbl.setStyleSheet("font-size: 16px; font-weight: bold; margin-bottom: 10px;")
        control_panel.addWidget(lbl)
        
        self.g_slider = QtWidgets.QSlider(QtCore.Qt.Orientation.Horizontal)
        self.g_slider.setRange(0, 200)
        self.g_slider.setValue(100) # Default Interaction g=100
        
        control_panel.addWidget(QtWidgets.QLabel("Interaction Strength (g)"))
        control_panel.addWidget(self.g_slider)
        
        layout.addLayout(control_panel, stretch=1)

        # --- Right Panel: High Speed PyQtGraph Rendering ---
        self.glw = pg.GraphicsLayoutWidget()
        layout.addWidget(self.glw, stretch=4)
        
        # Density View
        view_dens = self.glw.addViewBox(row=0, col=0, title="Density")
        view_dens.setAspectLocked(True)
        self.img_dens = pg.ImageItem()
        view_dens.addItem(self.img_dens)
        
        # Phase View
        view_phase = self.glw.addViewBox(row=0, col=1, title="Phase")
        view_phase.setAspectLocked(True)
        self.img_phase = pg.ImageItem()
        view_phase.addItem(self.img_phase)

        # Set Scientific Colormaps
        # PyQtGraph uses a different lookup table format than Matplotlib
        colormap_dens = pg.colormap.get('magma')
        self.img_dens.setLookupTable(colormap_dens.getLookupTable())
        
        # 'hsv' is built-in and acts identically to twilight for phase wrapping
        colormap_phase = pg.colormap.get('hsv') 
        self.img_phase.setLookupTable(colormap_phase.getLookupTable())

        self.win.show()

    def update_loop(self):
        # 1. Grab interaction variable from UI
        g = cp.float32(self.g_slider.value())
        psi = self.psi
        exp_K = self.exp_K
        half_dt = 0.5 * self.dt
        
        # Precompute scalars to avoid redundant GPU math
        g_hdt = -1j * half_dt * g
        g_dt = -1j * self.dt * g

        # 2. RUN PHYSICS ENTIRELY ON GPU
        dens = psi.real**2 + psi.imag**2
        psi *= cp.exp(g_hdt * dens)
        
        for _ in range(self.steps_per_frame - 1):
            psi_k = cufft.fft2(psi, overwrite_x=True)
            psi_k *= exp_K
            psi = cufft.ifft2(psi_k, overwrite_x=True)
            
            dens = psi.real**2 + psi.imag**2
            psi *= cp.exp(g_dt * dens)
                
        psi_k = cufft.fft2(psi, overwrite_x=True)
        psi_k *= exp_K
        psi = cufft.ifft2(psi_k, overwrite_x=True)
        
        dens = psi.real**2 + psi.imag**2
        psi *= cp.exp(g_hdt * dens)
            
        self.psi = psi

        # 3. PULL DATA TO CPU FOR RENDERING
        # PyQtGraph expects dimensions in (Width, Height), so we transpose (.T)
        dens_cpu = dens.get().T 
        
        # Update Density Image
        self.img_dens.setImage(dens_cpu, autoLevels=False, levels=(0, self.max_dens))
        
        # Update Phase Image (Masked by density to hide quantum vacuum noise)
        phase_cpu = cp.angle(self.psi).get().T
        mask_cpu = dens_cpu > (0.01 * self.max_dens)
        phase_cpu[~mask_cpu] = np.nan
        
        self.img_phase.setImage(phase_cpu, autoLevels=False, levels=(-np.pi, np.pi))

if __name__ == '__main__':
    app = GPE_GPU_MVP()
    sys.exit(app.app.exec())

ImportError: 
================================================================
Failed to import CuPy.

If you installed CuPy via wheels (cupy-cudaXXX or cupy-rocm-X-X), make sure that the package matches with the version of CUDA or ROCm installed.

On Linux, you may need to set LD_LIBRARY_PATH environment variable depending on how you installed CUDA/ROCm.
On Windows, try setting CUDA_PATH environment variable.

Check the Installation Guide for details:
  https://docs.cupy.dev/en/latest/install.html

CUDA Path: None
DLL dependencies:
  KERNEL32.dll -> C:\WINDOWS\System32\KERNEL32.dll
  MSVCP140.dll -> C:\Users\Wenca\AppData\Roaming\Python\Python312\site-packages\pyzmq.libs\MSVCP140.dll
  VCRUNTIME140.dll -> c:\Users\Wenca\AppData\Local\Programs\Python\Python312\VCRUNTIME140.dll
  VCRUNTIME140_1.dll -> c:\Users\Wenca\AppData\Local\Programs\Python\Python312\VCRUNTIME140_1.dll
  api-ms-win-crt-convert-l1-1-0.dll -> C:\WINDOWS\System32\ucrtbase.dll
  api-ms-win-crt-environment-l1-1-0.dll ->

In [10]:
!python -m pip uninstall numpy scipy cupy cupy-cuda13x cupy-cuda12x -y

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.17.1
Uninstalling scipy-1.17.1:
  Successfully uninstalled scipy-1.17.1
Found existing installation: cupy-cuda13x 13.6.0
Uninstalling cupy-cuda13x-13.6.0:
  Successfully uninstalled cupy-cuda13x-13.6.0


You can safely remove it manually.
You can safely remove it manually.


In [ ]:
!python -m pip install numpy==1.26.4 scipy cupy-cuda12x --no-cache-dir